# Quantum phase transitions with Qiskit
### Finding the ground state of the Ising chain with a variational quantum algorithm

In this tutorial you will use a quantum algorithm, the **variational quantum eigensolver (VQE)**, to study a **quantum phase transition** in a chain of qubits. You will:

1. write the Hamiltonian of the Ising model with Qiskit;
2. find its lowest-energy state, the **ground state**, with VQE;
3. measure how magnetized the chain is, and watch the phase transition appear, and get sharper, as the chain gets longer.

**What you need to know:** basic Python, and the basics of qubits and of the Pauli matrices $X$ and $Z$.
**Software:** Qiskit 2.x, NumPy, SciPy and Matplotlib.

*Adapted from the PennyLane demo [“Seeing quantum phase transitions with quantum computers”](https://pennylane.ai/qml/demos/tutorial_quantum_phase_transitions). Rewritten for Qiskit and adapted for this hackaton. The figures are from the original demo.*

## How to use this notebook

Most of the code is given, but some key pieces are left to you.

- **Tasks** (Tasks 1 to 7, plus the optional Tasks 8 and 9 in Section 8) ask you to write code where you see `# YOUR CODE HERE`. Keep the names and the arguments of the functions as they are: later cells use them.
- After most tasks, a **check** cell tests your code and tells you what is wrong, if anything.
- If you finish early, the last section asks you to adapt the exercise to a two-dimensional lattice.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.sparse.linalg import eigsh

import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import efficient_su2
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.quantum_info import SparsePauliOp, Statevector

print("Qiskit version:", qiskit.__version__)

# One color and one marker per chain length, used in every figure
SIZE_COLORS = {4: "#2a78d6", 5: "#eb6834", 6: "#1baf7a", 8: "#eda100", 10: "#e87ba4"}
SIZE_MARKERS = {4: "o", 5: "x", 6: "s", 8: "^", 10: "D"}
plt.rcParams.update({"figure.figsize": (6.4, 4.2), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

The next cell defines the functions that check your work in the tasks. Run it; you don't need to read it.

In [2]:
# Functions that check your work in the tasks. Run this cell; you don't need to read it.
from qiskit.quantum_info import Operator

_PAULI = {"I": np.eye(2), "X": np.array([[0.0, 1.0], [1.0, 0.0]]), "Z": np.diag([1.0, -1.0])}
_REFERENCE_J = [0.0, 0.25, 0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0, 5.0, 7.5]
_REFERENCE_E = {  # VQE energies of the reference solution, for the values of J above
    4: [-4.0000, -4.0468, -4.1865, -4.4169, -4.5972, -4.7342, -4.8843, -5.1330, -5.6056, -6.7663, -15.3010, -22.7000],
    5: [-5.0000, -5.0624, -5.2490, -5.5575, -5.7997, -5.9842, -6.1868, -6.5234, -7.1773, -8.8770, -20.3499, -30.2333],
    6: [-6.0000, -6.0781, -6.3115, -6.6982, -7.0022, -7.2342, -7.4893, -7.9195, -8.8748, -11.0015, -25.4012, -37.7666],
    8: [-8.0000, -8.1093, -8.4365, -8.9794, -9.4072, -9.7342, -10.0943, -10.8342, -12.2172, -15.2514, -35.5000, -52.8333]
}


def _fail(message):
    raise AssertionError(message)


def _pauli_matrix(N, paulis):
    # Matrix of a product of Pauli operators, e.g. {0: "Z", 1: "Z"}; qubit 0 is the rightmost factor, as in Qiskit
    matrix = np.array([[1.0]])
    for q in reversed(range(N)):
        matrix = np.kron(matrix, _PAULI[paulis.get(q, "I")])
    return matrix


def _ising_matrix(N, J, h_x=0.0):
    zz = sum(_pauli_matrix(N, {i: "Z", i + 1: "Z"}) for i in range(N - 1))
    x = sum(_pauli_matrix(N, {i: "X"}) for i in range(N))
    return -J * zz - h_x * x


def _magnetization_matrix(N):
    return sum(_pauli_matrix(N, {i: "Z"}) for i in range(N)) / N


def _exact(N, J, h_x):
    # Exact ground-state energy and sqrt(<M^2>)
    values, vectors = np.linalg.eigh(_ising_matrix(N, J, h_x))
    psi, M = vectors[:, 0], _magnetization_matrix(N)
    return values[0], np.sqrt(np.real(psi.conj() @ M @ M @ psi))


def check_hamiltonian(ising_hamiltonian, with_field=False):
    task = "Task 4" if with_field else "Task 1"
    for N in (2, 3, 4, 5):
        for J in (0.5, 2.0):
            for h_x in ((0.0, 0.7) if with_field else (None,)):
                call = f"ising_hamiltonian({N}, {J})" if h_x is None else f"ising_hamiltonian({N}, {J}, {h_x})"
                H = ising_hamiltonian(N, J) if h_x is None else ising_hamiltonian(N, J, h_x)
                if not isinstance(H, SparsePauliOp):
                    _fail(f"{task}: {call} returned {type(H).__name__}, not a SparsePauliOp. Did you forget the `return`?")
                if H.num_qubits != N:
                    _fail(f"{task}: {call} acts on {H.num_qubits} qubits instead of {N}.")
                if not np.allclose(H.to_matrix(), _ising_matrix(N, J, h_x or 0.0)):
                    _fail(f"{task}: {call} is not the right Hamiltonian. Check which qubits each term acts on, "
                          "and the sign and size of the coefficients.")
    print(f"{task} passed.")


def check_ansatz(create_ansatz):
    rng = np.random.default_rng(1)
    for N in (2, 3, 4):
        qc = create_ansatz(N)
        if not isinstance(qc, QuantumCircuit):
            _fail(f"Task 2: create_ansatz({N}) returned {type(qc).__name__}, not a QuantumCircuit. "
                  "Did you forget the `return`?")
        if qc.num_qubits != N:
            _fail(f"Task 2: create_ansatz({N}) has {qc.num_qubits} qubits instead of {N}.")
        expected = {"rz": 4 * N, "rx": 2 * N, "cx": N - 1}
        gates = {name: count for name, count in qc.count_ops().items() if name != "barrier"}
        if gates != expected:
            _fail(f"Task 2: create_ansatz({N}) contains the gates {gates}, but it should contain {expected}.")
        if qc.num_parameters != 6 * N:
            _fail(f"Task 2: create_ansatz({N}) uses {qc.num_parameters} different angles, but it should use "
                  f"6N = {6 * N}, each in one gate.")
        library = efficient_su2(N, su2_gates=["rz", "rx", "rz"], entanglement="linear", reps=1)
        angles = rng.uniform(0, 2 * np.pi, 6 * N)
        if not Operator(qc.assign_parameters(angles)).equiv(Operator(library.assign_parameters(angles))):
            _fail(f"Task 2: create_ansatz({N}) has the right gates, but it is not the same circuit as the library one. "
                  "Check the order of the gates, the control and the target of each CNOT, and which angle "
                  "goes in each gate.")
    print("Task 2 passed: your circuit is the same as the library one.")


def check_first_vqe(runs):
    if not (isinstance(runs, list) and len(runs) > 0 and all(isinstance(run, dict) and "E" in run for run in runs)):
        _fail("Task 3: `runs` should be what run_vqe returns: a list of runs, the best one first.")
    if abs(runs[0]["E"] + 4) > 1e-4:
        _fail(f"Task 3: the lowest energy found is {runs[0]['E']:.5f}, but it should be -4. Did you pass "
              "the ansatz of Task 2 and the Hamiltonian H of Task 1 (N = 3, J = 2)?")
    print("Task 3 passed.")


def check_magnetization(magnetization_op, magnetization_squared_op):
    for N in (2, 3, 5):
        M, M2 = magnetization_op(N), magnetization_squared_op(N)
        for name, op in (("magnetization_op", M), ("magnetization_squared_op", M2)):
            if not isinstance(op, SparsePauliOp):
                _fail(f"Task 5: {name}({N}) returned {type(op).__name__}, not a SparsePauliOp.")
        M_matrix = _magnetization_matrix(N)
        if not np.allclose(M.to_matrix(), M_matrix):
            _fail(f"Task 5: magnetization_op({N}) is not (1/N) * sum_i Z_i.")
        if not np.allclose(M2.to_matrix(), M_matrix @ M_matrix):
            _fail(f"Task 5: magnetization_squared_op({N}) is not M^2.")
        if len(M2) > 1 + N * (N - 1) // 2:
            print(f"Task 5: magnetization_squared_op({N}) is right, but it contains {len(M2)} Pauli strings. "
                  "Use .simplify() to collect the equal ones.")
    print("Task 5 passed.")


def check_scan(J_list, energies, m2_values, iterations, N, h_x, task="Task 6"):
    n = len(J_list)
    for name, values in (("energies", energies), ("<M^2> values", m2_values), ("iteration counts", iterations)):
        if len(values) != n:
            _fail(f"{task}: there are {len(values)} {name} for {n} values of J. "
                  "Is something appended twice, or not at all?")
    for J, E, m2 in zip(J_list, energies, m2_values):
        E0, exact_rms = _exact(N, J, h_x)
        if E < E0 - 1e-6:
            _fail(f"{task}: for N = {N}, J = {J} the energy {E:.4f} is below the exact ground-state energy "
                  f"{E0:.4f}, which is impossible. Is the Hamiltonian right, with h_x?")
        if E - E0 > 0.1 * abs(E0):
            _fail(f"{task}: for N = {N}, J = {J} the energy {E:.4f} is far above the exact value {E0:.4f}. "
                  "Did you measure the best run, runs[0]?")
        if not 0 <= m2 <= 1 + 1e-9 or abs(np.sqrt(max(m2, 0)) - exact_rms) > 0.25:
            _fail(f"{task}: for N = {N}, J = {J}, <M^2> = {m2:.4f} does not look right "
                  f"(the exact value is {exact_rms**2:.4f}). Did you measure M2?")
    reference = _REFERENCE_E.get(N)
    if reference is not None and list(J_list) == _REFERENCE_J and np.allclose(energies, reference, atol=1e-3):
        print(f"{task} passed for N = {N}: your results are the same as those of the reference solution.")
    else:
        print(f"{task} passed for N = {N}.")


def check_scan_dict(scan, J_list, h_x):
    for N in sorted(scan):
        if N == 5:
            continue  # from Task 6
        d = scan[N]
        if not isinstance(d, dict) or not {"runs", "E", "M2", "iterations"} <= set(d):
            _fail(f"Task 7: scan[{N}] should be a dictionary with the keys 'runs', 'E', 'M2' and 'iterations'.")
        check_scan(J_list, d["E"], d["M2"], d["iterations"], N, h_x, task="Task 7")


# VQE values of U_4 in the reference solution (one layer), for the values of J of _REFERENCE_J
_REFERENCE_U4_VQE = {
    4: [0.167, 0.232, 0.301, 0.369, 0.407, 0.432, 0.455, 0.488, 0.534, 0.630, 0.662, 0.665],
    6: [0.111, 0.161, 0.218, 0.283, 0.326, 0.355, 0.384, 0.571, 0.613, 0.644, 0.663, 0.665],
    8: [0.083, 0.123, 0.169, 0.226, 0.266, 0.295, 0.327, 0.595, 0.626, 0.650, 0.664, 0.666],
}


def _exact_binder(N, J, h_x):
    # U_4 in the exact ground state
    psi = np.linalg.eigh(_ising_matrix(N, J, h_x))[1][:, 0]
    p, m = np.abs(psi) ** 2, np.diag(_magnetization_matrix(N))
    m2, m4 = p @ m**2, p @ m**4
    return 1 - m4 / (3 * m2**2)


def _check_binder_dict(task, name, values, sizes, J_values):
    if not isinstance(values, dict) or sorted(values) != sorted(sizes):
        _fail(f"{task}: {name} should be a dictionary with one entry for each N in {list(sizes)}, "
              f"but it has the keys {sorted(values) if isinstance(values, dict) else type(values).__name__}.")
    for N in sizes:
        U = np.asarray(values[N])
        if U.shape != (len(J_values),):
            _fail(f"{task}: {name}[{N}] should contain one value for each of the {len(J_values)} values of J, "
                  f"but it has the shape {U.shape}.")
        if np.iscomplexobj(U):
            _fail(f"{task}: {name}[{N}] contains complex numbers. Expectation values are real: use .real.")


def check_binder_exact(J_fine, binder_sizes, binder, h_x):
    task = "Task 8"
    _check_binder_dict(task, "binder", binder, binder_sizes, J_fine)
    for N in binder_sizes:
        for k in ((0, 45, 90) if N >= 10 else (0, 30, 45, 60, 90)):
            U_exact = _exact_binder(N, J_fine[k], h_x)
            if abs(binder[N][k] - U_exact) > 1e-6:
                _fail(f"{task}: for N = {N} and J = {J_fine[k]:.3f}, U_4 should be {U_exact:.4f}, "
                      f"but binder[{N}] gives {np.real(binder[N][k]):.4f}.")
    print(f"{task} passed.")


def check_binder_vqe(J_list, vqe_sizes, vqe_binder, h_x):
    task = "Task 9"
    _check_binder_dict(task, "vqe_binder", vqe_binder, vqe_sizes, J_list)
    if list(J_list) != _REFERENCE_J:
        print(f"{task}: the structure is right (J_list differs from the reference, so the values are not checked).")
        return
    k = _REFERENCE_J.index(1.0)
    for N in vqe_sizes:
        if N in _REFERENCE_U4_VQE and abs(vqe_binder[N][k] - _REFERENCE_U4_VQE[N][k]) > 2e-3:
            if abs(vqe_binder[N][k] - _exact_binder(N, 1.0, h_x)) < 1e-3:
                _fail(f"{task}: vqe_binder[{N}] contains the values of the exact ground states. Task 9 asks for "
                      "U_4 in the VQE states: the circuit with the optimized angles stored in scan.")
    if all(np.allclose(vqe_binder[N], _REFERENCE_U4_VQE[N], atol=2e-3) for N in vqe_sizes if N in _REFERENCE_U4_VQE):
        print(f"{task} passed: your values are the same as those of the reference solution.")
    else:
        print(f"{task}: the structure is right, but the values differ from those of the reference solution. If your "
              "scans of Tasks 6 and 7 matched the reference, check that you used the best run, runs[0], the circuit "
              "of the scans, and M^4 = M^2 M^2.")


## 1. What is a phase transition?

A **phase transition** is an abrupt change in the properties of a system when one of its parameters is varied. Water freezing into ice when the temperature drops below 0 °C is the everyday example. Phase transitions appear everywhere in physics, from magnets and superconductors to the early universe.

A **quantum** phase transition happens at zero temperature. It is driven not by heat but by *quantum fluctuations*, a consequence of Heisenberg's uncertainty principle: changing a parameter in the Hamiltonian changes the nature of the ground state. For large systems these transitions may be very hard to compute on classical computers (it is not just a matter of size, it generally depends also on the degree of entanglement in the system), which is one reason why physicists want to study them on quantum computers.

Today's exercise is in the proof of concept domain: we won't do anything a classical computer couldn't do efficiently, it is a simple example with didactical purposes.

## 2. The Ising model

The simplest Ising model is a chain of $N$ qubits on a line. Think of each qubit as a small magnet (a *spin*) that can point up ($|0\rangle$) or down ($|1\rangle$) along $z$. Each spin interacts with its neighbours on either side.

<center><img src="https://raw.githubusercontent.com/PennyLaneAI/demos/a83efb73d7989aa60917124c25ce0469c02bae02/_static/demonstration_assets/quantum_phase_transitions/Fig_1_Ising_chain.png" width="30%" alt="A chain of qubits, each interacting with its nearest neighbours"></center>

The energy of the chain is given by the Hamiltonian

$$H = -J \sum_{i=0}^{N-2} Z_i Z_{i+1},$$

where $Z_i$ is the Pauli-$Z$ operator acting on qubit $i$ and $J > 0$ is the strength of the interaction. Each term gives $-J$ when two neighbours point the same way and $+J$ when they point opposite ways, so the energy is lowest when all spins agree.

In Qiskit, Hamiltonians are written as sums of *Pauli strings* with `SparsePauliOp`. Note that Qiskit prints the strings with qubit 0 on the **right**: `"IZZ"` means $Z$ on qubits 0 and 1, and nothing (the identity $I$) on qubit 2. In Task 1 you write this Hamiltonian as a `SparsePauliOp`; in Section 4 you will add a second term to it.

>#### Qiskit note: `SparsePauliOp`
>
>A `SparsePauliOp` stores an operator as a sum of Pauli strings with coefficients, $H = \sum_k c_k P_k$. `SparsePauliOp.from_sparse_list(terms, num_qubits)` builds it from a list of tuples `(Pauli string, qubit indices, coefficient)`: for example `("ZZ", [1, 2], -2)` is the term $-2\,Z_1 Z_2$, and every qubit not listed gets the identity $I$. [Full documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.SparsePauliOp).

#### Task 1: the Hamiltonian as Pauli strings

Complete `ising_hamiltonian(N, J)`, so that it returns $H = -J \sum_{i=0}^{N-2} Z_i Z_{i+1}$ as a `SparsePauliOp` for a chain of any length $N$. Build the list of terms, one for each pair of neighbours, and turn it into an operator with `SparsePauliOp.from_sparse_list` (Important: use from_sparse_list and not from_list if you want to just deal with the affected qubits and avoid the need for Identities, e.g. for N=3: (ZZ, [1,2], some_coeff) vs (ZZI, some_coeff) ).

For $N = 3$ and $J = 2$ you should get the two Pauli strings `IZZ` and `ZZI`, each with the coefficient $-2$.

In [17]:
def ising_hamiltonian(N, J):
    # YOUR CODE HERE
    # suggestions: 
    # terms = []
    # for i in some_meaningful_range:
    #   append terms one-by-one
    # return the operator (see the intro notebook if confused)

In [21]:
N = 3
J = 2

H = ising_hamiltonian(N, J)
print(f"H = {H}")
check_hamiltonian(ising_hamiltonian)

H = SparsePauliOp(['IZZ', 'ZZI'],
              coeffs=[-2.+0.j, -2.+0.j])
Task 1 passed.


#### Question

Before we use any algorithm: for $N = 3$ and $J = 2$, which states of the chain have the lowest energy, and what is that energy? *Hint:* what does each term $-2\,Z_i Z_{i+1}$ give when the two spins point the same way, and when they don't?

## 3. Finding the ground state with VQE

The **ground state** is the state of lowest energy. For three qubits we can guess it, but in general we need an algorithm. The **variational quantum eigensolver (VQE)** works like this:

1. A quantum circuit with adjustable rotation angles $\boldsymbol\theta$ prepares a trial state $|\psi(\boldsymbol\theta)\rangle$. This circuit is called the **ansatz**.
2. The quantum computer measures the energy $E(\boldsymbol\theta) = \langle\psi(\boldsymbol\theta)|H|\psi(\boldsymbol\theta)\rangle$.
3. A classical optimizer changes $\boldsymbol\theta$ to lower the energy, and steps 2 and 3 are repeated.

The **variational principle** guarantees that $E(\boldsymbol\theta)$ is never below the true ground-state energy, so the lowest energy we find is our best estimate of it.

As ansatz we use the **hardware-efficient ansatz** [2], which is easy to run on real devices:

1. three rotations ($R_Z$, $R_X$, $R_Z$) on each qubit, each with its own angle;
2. a CNOT gate between each pair of neighbouring qubits;
3. another three rotations ($R_Z$, $R_X$, $R_Z$) on each qubit.

That makes $6N$ angles in total.

Qiskit's circuit library contains this circuit: `efficient_su2` builds it for us. Here it is for $N = 3$ qubits:

In [22]:
def use_premade_ansatz(N):
    # The hardware-efficient ansatz from Qiskit's circuit library
    return efficient_su2(N, su2_gates=["rz", "rx", "rz"], entanglement="linear", reps=1)


print(use_premade_ansatz(3).draw("text", fold=-1))

     ┌──────────┐┌──────────┐┌──────────┐     ┌──────────┐┌───────────┐┌───────────┐             
q_0: ┤ Rz(θ[0]) ├┤ Rx(θ[3]) ├┤ Rz(θ[6]) ├──■──┤ Rz(θ[9]) ├┤ Rx(θ[12]) ├┤ Rz(θ[15]) ├─────────────
     ├──────────┤├──────────┤├──────────┤┌─┴─┐└──────────┘├───────────┤├───────────┤┌───────────┐
q_1: ┤ Rz(θ[1]) ├┤ Rx(θ[4]) ├┤ Rz(θ[7]) ├┤ X ├─────■──────┤ Rz(θ[10]) ├┤ Rx(θ[13]) ├┤ Rz(θ[16]) ├
     ├──────────┤├──────────┤├──────────┤└───┘   ┌─┴─┐    ├───────────┤├───────────┤├───────────┤
q_2: ┤ Rz(θ[2]) ├┤ Rx(θ[5]) ├┤ Rz(θ[8]) ├────────┤ X ├────┤ Rz(θ[11]) ├┤ Rx(θ[14]) ├┤ Rz(θ[17]) ├
     └──────────┘└──────────┘└──────────┘        └───┘    └───────────┘└───────────┘└───────────┘


#### Task 2: build the ansatz yourself

Complete `create_ansatz(N)`, which builds the same circuit for any $N$, gate by gate, with `qc.rz`, `qc.rx` and `qc.cx` (intro notebook, Sections 3, 5 and 10). Look at the drawing to see which angle `theta[k]` goes in each gate, and work out the rule for any $N$.

The numbering matters: the check compares your circuit with the library one for random angles, and the results later in the notebook depend on it.

<details>
<summary>Hint (click to open)</summary>

On qubit `i`, the first three rotations use `theta[i]`, `theta[N + i]` and `theta[2 * N + i]`, and the last three use `theta[3 * N + i]`, `theta[4 * N + i]` and `theta[5 * N + i]`. Each CNOT has qubit `i` as control and qubit `i + 1` as target.
</details>

In [ ]:
def create_ansatz(N):
    # Hardware-efficient ansatz with 6N parameters θ[0], ..., θ[6N-1]
    theta = ParameterVector("θ", 6 * N)
    qc = QuantumCircuit(N)
    # STEP 1: single-qubit rotations RZ, RX, RZ on all the qubits
    # YOUR CODE HERE

    # STEP 2: a CNOT gate on each pair of neighbouring qubits
    # YOUR CODE HERE

    # STEP 3: single-qubit rotations RZ, RX, RZ on all the qubits
    # YOUR CODE HERE

    return qc

In [ ]:
ansatz = create_ansatz(N)
print(ansatz.draw("text", fold=-1))
check_ansatz(create_ansatz)

### Measuring energies and optimizing the angles

- **Measuring the energy.** In Qiskit, expectation values like $E(\boldsymbol\theta)$ are computed by an **Estimator**. You give it a circuit, an observable (here $H$) and values for the angles. We use the Estimator of **Qiskit Aer**, a fast compiled simulator, which computes the expectation values exactly. It works like the `StatevectorEstimator` of the intro notebook, and gives the same results about five times faster.
- **Which way is downhill?** The optimizer also needs the *gradient*: how the energy changes when each angle changes. On a quantum computer this is obtained with the **parameter-shift rule**: each derivative is the difference between two energies, measured with that angle shifted by $+\pi/2$ and $-\pi/2$. We send all these circuits to the Estimator at once.
- **The optimizer.** We use **L-BFGS-B** from SciPy, a standard method for smooth functions. It uses the gradients it has already seen to choose good steps, and stops when the energy can no longer be lowered.
- **Several starting points.** An optimizer can get stuck in a *local minimum*, a valley that is not the lowest one. To be safe, we start it from three random sets of angles and keep the result with the lowest energy.

> *For the curious:* for rotation gates such as $R_X$ and $R_Z$, the parameter-shift rule is exact:
> $\dfrac{\partial E}{\partial \theta_k} = \dfrac{E(\boldsymbol\theta + \frac{\pi}{2}\mathbf e_k) - E(\boldsymbol\theta - \frac{\pi}{2}\mathbf e_k)}{2}$.

The two functions below are given. In the tasks you will call `run_vqe(ansatz, hamiltonian, rng)`: it returns the three runs, the best one first. Each run is a dictionary; the most useful entries are the optimized angles `run["params"]`, the final energy `run["E"]` and the number of iterations `run["iterations"]`.

In [ ]:
# Qiskit Aer's Estimator: a fast compiled simulator, which computes the expectation values exactly
estimator = AerEstimator()


def make_cost_function(ansatz, hamiltonian):
    # Returns a function params -> (energy, gradient), with the gradient from the parameter-shift rule
    n_params = ansatz.num_parameters
    shifts = (np.pi / 2) * np.eye(n_params)

    def energy_and_gradient(params):
        # all circuits in one batch: the angles themselves, then each angle shifted by +π/2, then by -π/2
        batch = np.vstack([params, params + shifts, params - shifts])
        evs = estimator.run([(ansatz, hamiltonian, batch)]).result()[0].data.evs
        return evs[0], 0.5 * (evs[1:n_params + 1] - evs[n_params + 1:])

    return energy_and_gradient


def run_vqe(ansatz, hamiltonian, rng, n_starts=3):
    # VQE with SciPy's L-BFGS-B, started from n_starts random sets of angles.
    # Returns all runs, the one with the lowest energy first. Each run records its start number,
    # the optimized angles, the energy after every iteration, the final energy and the number of iterations.
    cost = make_cost_function(ansatz, hamiltonian)
    runs = []
    for start in range(n_starts):
        params = rng.uniform(0, 2 * np.pi, ansatz.num_parameters)
        energy = [estimator.run([(ansatz, hamiltonian, params)]).result()[0].data.evs.item()]
        result = minimize(cost, params, jac=True, method="L-BFGS-B",
                          callback=lambda intermediate_result: energy.append(intermediate_result.fun))
        runs.append({"start": start, "params": result.x, "energy": energy, "E": result.fun, "iterations": result.nit})
    return sorted(runs, key=lambda run: run["E"])

>#### Library note: what `run_vqe` is made of
>
>`run_vqe` is short because two libraries do the real work.
>
>- **Qiskit's Estimator**, here the [`EstimatorV2` of Qiskit Aer](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.EstimatorV2.html), computes the energies: `estimator.run([(circuit, observable, parameter_values)])` evaluates the observable for one or many sets of angles at once (intro notebook, Section 9). `make_cost_function` uses it to get the energy and, with the parameter-shift rule, the gradient, in a single call.
>- **SciPy's [`minimize`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)** changes the angles to lower the energy. `method="L-BFGS-B"` chooses the algorithm ([its options](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-lbfgsb.html), for example when it stops), `jac=True` tells it that the cost function returns the gradient as well as the energy, and `callback` is called after every iteration, here to record the energy. It returns the optimized angles `result.x`, the final energy `result.fun` and the number of iterations `result.nit`.
>
>`run_vqe` calls `minimize` once for each random starting point and sorts the runs by energy: the same recipe as the tiny VQE in Section 12 of the intro notebook.

### A first example: three qubits

Let's find the ground state for $N = 3$ and $J = 2$. The random starting angles come from a generator with a fixed *seed*, so you get the same numbers every time you run the notebook.

#### Task 3: launch VQE

Now run VQE yourself: call `run_vqe` for the circuit `ansatz` (Task 2) and the Hamiltonian `H` (Task 1), with the random generator `rng`, and store what it returns in a variable called `runs`. Look at the definition of `run_vqe` above to see which arguments it takes and what it returns. It takes a few seconds.

In [ ]:
rng = np.random.default_rng(3)  # fixed seed: reproducible starting angles

# YOUR CODE HERE: run VQE and store the result in `runs`
# runs = run_vqe(put here the correct arguments)

In [ ]:
check_first_vqe(runs)

energy = runs[0]["energy"]  # energy history of the best run, plotted below
for run in sorted(runs, key=lambda run: run["start"]):
    print(f"start {run['start']}: {run['iterations']:3d} L-BFGS-B iterations, final energy E = {run['E']:.5f}")

In [ ]:
plt.plot(range(len(energy)), energy, color="0.2", lw=2, marker="o", ms=4)
plt.axhline(-4, color="0.5", lw=1, ls="--")
plt.text(len(energy) - 1, -3.8, "E = −4", ha="right", va="bottom", color="0.35")
plt.xlabel("L-BFGS-B iteration")
plt.ylabel("Energy")
plt.title("VQE for the Ising chain, N = 3, J = 2")
plt.show()

All three starts reach $E = -4$, in fewer than 20 iterations.

But which state has VQE found? On a simulator we can look at the state itself: `Statevector(circuit)` computes the full quantum state prepared by a circuit, and `probabilities_dict()` gives the probability of each measurement outcome. (On a real quantum computer the state is not accessible: we would have to estimate these probabilities by measuring many times.)

The helper below writes a state as a sum of basis states, and the cell after it prints the state found by each of the three runs, built from its optimized angles with `ansatz.assign_parameters`.

In [ ]:
def ket_string(state, tol=0.01):
    # Writes a state as a sum of basis states
    # Multiplying a state by a global phase e^(iφ) changes nothing physical, so we pick the phase
    # that makes the largest amplitude real and positive
    amplitudes = state.to_dict()  # {"000": amplitude, "001": amplitude, ...}
    largest = max(amplitudes.values(), key=abs)
    terms = []
    for bits, a in amplitudes.items():
        a = a * abs(largest) / largest  # remove the global phase
        if abs(a) > tol:
            coefficient = f"{a.real:.3f}" if abs(a.imag) < 5e-4 else f"({a.real:.3f}{a.imag:+.3f}i)"
            terms.append(f"{coefficient}|{bits}⟩")
    return " + ".join(terms).replace("+ -", "- ")

In [ ]:
for run in sorted(runs, key=lambda run: run["start"]):
    psi = Statevector(ansatz.assign_parameters(run["params"]))  # the state prepared with the optimized angles
    probabilities = psi.probabilities_dict()
    print(f"start {run['start']}:  |ψ⟩ = {ket_string(psi)}")
    print(f"          P(000) = {probabilities.get('000', 0):.3f},  P(111) = {probabilities.get('111', 0):.3f}")

## 4. Adding a transverse field

Now we add a magnetic field along $x$, perpendicular to the $z$ direction of the interaction. It is called a **transverse field**, and it adds the term $-h_x \sum_i X_i$:

<center><img src="https://raw.githubusercontent.com/PennyLaneAI/demos/a83efb73d7989aa60917124c25ce0469c02bae02/_static/demonstration_assets/quantum_phase_transitions/Fig_2_transverse_Ising.png" width="30%" alt="The Ising chain in a transverse magnetic field"></center>

$$H = -J \sum_{i=0}^{N-2} Z_i Z_{i+1} \;-\; h_x \sum_{i=0}^{N-1} X_i .$$

The two terms compete, like a tug of war:

- When $J \gg h_x$, the interaction wins and the spins all align along $z$. This is the **ordered** phase.

  <center><img src="https://raw.githubusercontent.com/PennyLaneAI/demos/a83efb73d7989aa60917124c25ce0469c02bae02/_static/demonstration_assets/quantum_phase_transitions/Fig_3_ground_state_J_large.png" width="50%" alt="Ground state for J much larger than h_x: all spins along z"></center>

- When $h_x \gg J$, the field wins and every spin points along $x$, parallel to the field. Along $z$ they are then completely random. This is the **disordered** phase.

  <center><img src="https://raw.githubusercontent.com/PennyLaneAI/demos/a83efb73d7989aa60917124c25ce0469c02bae02/_static/demonstration_assets/quantum_phase_transitions/Fig_4_ground_state_h_large.png" width="50%" alt="Ground state for h_x much larger than J: all spins along x"></center>

For an infinitely long chain, the switch between the two phases happens sharply at $J/h_x = 1$ [3, 4]. This is the quantum phase transition we want to see.

#### Task 4: add the transverse field

Extend `ising_hamiltonian` with a third argument `h_x`, with the default value 0, which adds the term $-h_x \sum_{i=0}^{N-1} X_i$: an $X$ on each qubit, with the coefficient $-h_x$. With `h_x = 0` it must give the same Hamiltonian as before.

In [ ]:
def ising_hamiltonian(N, J, h_x=0.0):
    # Open chain:  H = -J * sum_i Z_i Z_{i+1}  -  h_x * sum_i X_i
    # YOUR CODE HERE (start from your solution of Task 1)
    # ...

In [ ]:
print("H for N = 3, J = 2, h_x = 1:", ising_hamiltonian(3, 2, h_x=1))
check_hamiltonian(ising_hamiltonian, with_field=True)

## 5. Measuring order: the magnetization

To tell the two phases apart we look at the **magnetization**, the average of all the $Z_i$:

$$M = \frac{1}{N} \sum_i Z_i .$$

If you measure all the qubits, $M$ is close to $+1$ or $-1$ in the ordered phase (almost all spins agree), and close to $0$ in the disordered phase (the spins are random).

>**A subtlety.** For a chain of finite length, the ordered ground state is not "all up" or "all down", but a *superposition* of the two, roughly $(|00\cdots0\rangle + |11\cdots1\rangle)/\sqrt2$. Each measurement finds an ordered chain, but up and down are equally likely, so the *average* $\langle M\rangle$ is zero, at every $J$. That's why we measure the average of $M^2$ instead: squaring removes the sign. Its square root, $\sqrt{\langle M^2\rangle}$, is the typical size of the magnetization:
>
>- it is close to 1 in the ordered phase;
>- it is small in the disordered phase, but not exactly zero for a short chain: even completely random spins give $\sqrt{\langle M^2\rangle} = 1/\sqrt N$.

$M^2 = \frac{1}{N^2} \sum_{i,j} Z_i Z_j$ is again a sum of Pauli strings, so the Estimator can measure it just like the energy.

For small chains we can also compute everything **exactly**, by writing $H$ as a $2^N \times 2^N$ matrix and finding its lowest eigenvector. This is impossible for large $N$, but here it lets us check the VQE results.

#### Task 5: the magnetization

Complete `magnetization_op(N)`, which returns $M = \frac{1}{N}\sum_i Z_i$, and `magnetization_squared_op(N)`, which returns $M^2$. You did this for $N = 3$ in Exercise 2 of the intro notebook. *Hint:* `@` multiplies two operators, and `.simplify()` collects equal Pauli strings.

In [ ]:
def magnetization_op(N):
    # M = (1/N) * sum_i Z_i
    # YOUR CODE HERE
    ...


def magnetization_squared_op(N):
    # M^2 = (1/N^2) * sum_ij Z_i Z_j
    # YOUR CODE HERE
    ...

In [ ]:
print("M^2 for N = 3:", magnetization_squared_op(3))
check_magnetization(magnetization_op, magnetization_squared_op)

The next two cells do this exact computation. `hamiltonian.to_matrix(sparse=True)` gives the $2^N \times 2^N$ matrix of $H$ as a *sparse* matrix, which stores only its non-zero entries, and SciPy's `eigsh` finds its lowest eigenvalue and the corresponding eigenvector. `exact_rms_magnetization` then computes $\sqrt{\langle M^2\rangle}$ in this exact ground state.

In [ ]:
def exact_ground_state(hamiltonian):
    # Lowest energy and corresponding state, by exact diagonalization of the 2^N x 2^N matrix (small N only)
    values, vectors = eigsh(hamiltonian.to_matrix(sparse=True), k=1, which="SA")
    return values[0], vectors[:, 0]

In [ ]:
def exact_rms_magnetization(N, J, h_x=1.0):
    # sqrt(<M^2>) in the exact ground state
    psi = Statevector(exact_ground_state(ising_hamiltonian(N, J, h_x))[1])
    return np.sqrt(psi.expectation_value(magnetization_squared_op(N)).real)

## 6. Scanning $J$ for a chain of 5 qubits

We fix the field at $h_x = 1$ and vary $J$, so that $J/h_x$ goes from 0 (field only) to 7.5 (interaction dominates), with more points around the expected transition at $J = 1$. For each value of $J$ we run VQE (three starts, best one kept) and measure the energy and $\langle M^2\rangle$ of the result in a single Estimator call.

#### Task 6: the scan

Complete the loop below. For each value of $J$:

1. build the Hamiltonian;
2. run VQE with `run_vqe`;
3. measure the energy (use the Hamiltonian operator itself) and $\langle M^2\rangle$ of the best state with a **single** Estimator call;
4. store the results in the four lists, which the following cells use.

*Hint:* an Estimator PUB can contain several observables: `estimator.run([(circuit, [obs1, obs2], params)]).result()[0].data.evs` gives one value for each (intro notebook, Section 9).

The scan takes about half a minute.

In [ ]:
N = 5

# h_x is the strength of the transverse magnetic field
h_x = 1

# the values of J (and therefore of J/h_x) that we scan
J_list = [0.0, 0.25, 0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0, 5.0, 7.5]

ansatz_5 = create_ansatz(N)
M2 = magnetization_squared_op(N)
rng = np.random.default_rng(N)  # fixed seed: reproducible starting angles

runs_5, energy_list, m2_list, iterations_list = [], [], [], []
for J in J_list:
    # YOUR CODE HERE
    # hint: look inside the previous run_vqe to get a feeling of how estimator.run works
    # ...
    # E, m2 = estimator.run(insert the correct arguments)
    # and append meaningful quantities
    # ...

In [ ]:
print(f"{'J':>5} {'iterations':>11} {'E':>10} {'sqrt<M2>':>9} {'exact':>7}")
for J, n_it, E, m2 in zip(J_list, iterations_list, energy_list, m2_list):
    print(f"{J:5.2f} {n_it:11d} {E:10.4f} {np.sqrt(m2):9.3f} {exact_rms_magnetization(N, J, h_x):7.3f}")

check_scan(J_list, energy_list, m2_list, iterations_list, N, h_x)

In [ ]:
J_dense = np.linspace(0, 7.5, 151)
plt.plot(J_dense, [exact_rms_magnetization(N, J, h_x) for J in J_dense], color=SIZE_COLORS[5], lw=1.5, alpha=0.6,
         label="exact (diagonalization)")
plt.plot(J_list, np.sqrt(m2_list), ls="none", marker="x", ms=8, mew=2, color=SIZE_COLORS[5], label="VQE")
plt.xlabel("J")
plt.ylabel(r"$\sqrt{\langle M^2 \rangle}$")
plt.title(r"$\sqrt{\langle M^2 \rangle}$ vs. $J$ for $N$=" + str(N) + r"  ($h_x = 1$)")
plt.legend(loc="lower right")
plt.show()

$\sqrt{\langle M^2\rangle}$ grows from $1/\sqrt5 \approx 0.45$ at $J = 0$ (random spins) to almost 1 at large $J$ (ordered spins). It rises most steeply around $J \approx 1$, where we expect the transition.

The VQE points (crosses) follow the exact curve closely. The small differences around $J \approx 1$–$1.5$ come from the circuit: with only one layer of CNOTs it cannot represent the true ground state perfectly there. Even so, every VQE energy is within 2% of the exact one (see the table in Section 7).

For 5 qubits the change is gradual rather than abrupt. A truly sharp transition only exists for an infinitely long chain. So what happens when the chain gets longer?

## 7. Longer chains: the transition gets sharper

We now repeat the scan for $N = 4$, 6 and 8 qubits, with the same values of $J$. For every point we also compute the exact values, for comparison. This is a longer test: it takes two to three minutes, mostly for $N = 8$.

#### Task 7: a scan for any $N$

Turn your loop of Task 6 into a function `magnetization_scan(N, J_list, h_x)`, which repeats the scan for a chain of $N$ qubits. It should return a dictionary with four entries, one value for each $J$: `"runs"` (a list), and `"E"`, `"M2"` and `"iterations"` (NumPy arrays). Create its own random generator, seeded with $N$ as in Task 6. The comments in the cell below give the steps in pseudocode.

The next cell runs it for $N$ = 4, 6 and 8. The three scans take two to three minutes, mostly for $N = 8$.

In [ ]:
def magnetization_scan(N, J_list, h_x=1.0):
    # YOUR CODE HERE
    # This is arguably the hardest task in the notebook, hence the structure is given, fill it correctly with the right functions
    #
    # ansatz = 
    # M2     =  
    # rng    =
    # out = {"runs": [], "E": [], "M2": [], "iterations": []}
    #
    # for J in ...
    #   H = theHamiltonian
    #   runs = use run_vqe
    #   E, m2 = 
    #   # here we store the results:
        for key, value in zip(out, (runs, E, m2, runs[0]["iterations"])):
            out[key].append(value)

    return {key: (value if key == "runs" else np.array(value)) for key, value in out.items()}

In [ ]:
sizes = [4, 6, 8]

scan = {5: {"runs": runs_5, "E": np.array(energy_list), "M2": np.array(m2_list),
            "iterations": np.array(iterations_list)}}
for N in sizes:
    scan[N] = magnetization_scan(N, J_list, h_x)
check_scan_dict(scan, J_list, h_x)

# exact ground-state energy and sqrt(<M^2>), for comparison
for N in scan:
    M2 = magnetization_squared_op(N)
    exact = [exact_ground_state(ising_hamiltonian(N, J, h_x)) for J in J_list]
    scan[N]["E0"] = np.array([e0 for e0, _ in exact])
    scan[N]["exact_rms"] = np.array([np.sqrt(Statevector(psi).expectation_value(M2).real) for _, psi in exact])

In [ ]:
sizes = sorted(scan)
print("For each N: sqrt<M^2> from VQE, the exact value, and the relative energy error of VQE |E_VQE - E0| / |E0|\n")
print(" " * 6 + "".join(f"{'N = ' + str(N):^26}" for N in sizes))
print(f"{'J':>5} " + "".join(f"{'VQE':>8}{'exact':>8}{'error':>8}  " for N in sizes))
for k, J in enumerate(J_list):
    row = f"{J:5.2f} "
    for N in sizes:
        d = scan[N]
        error = abs(d["E"][k] - d["E0"][k]) / abs(d["E0"][k])
        row += f"{np.sqrt(d['M2'][k]):8.3f}{d['exact_rms'][k]:8.3f}{error:8.2%}  "
    print(row)

In [ ]:
from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True, layout="constrained")
J_arr = np.array(J_list)
J_dense = np.linspace(0, 5, 101)
exact_dense = {N: np.array([exact_rms_magnetization(N, J, h_x) for J in J_dense]) for N in sizes}

for ax, J_max, title in zip(axes, (5.0, 2.0), ("J from 0 to 5", "Zoom on the transition region")):
    shown, dense = J_arr <= J_max, J_dense <= J_max
    for N in sizes:
        ax.plot(J_dense[dense], exact_dense[N][dense], color=SIZE_COLORS[N], lw=1.5, alpha=0.6)
        ax.plot(J_arr[shown], np.sqrt(scan[N]["M2"][shown]), ls="none", marker=SIZE_MARKERS[N], ms=7,
                color=SIZE_COLORS[N])
    ax.axvline(1.0, color="0.5", lw=1, ls=":")
    ax.set_xlabel("J")
    ax.set_title(title, fontsize=10)

axes[0].set_ylabel(r"$\sqrt{\langle M^2 \rangle}$")
axes[1].text(1.03, 0.98, r"$J/h_x = 1$", transform=axes[1].get_xaxis_transform(), color="0.35", fontsize=9, va="top")
handles = [Line2D([], [], color=SIZE_COLORS[N], marker=SIZE_MARKERS[N], lw=1.5, label=f"N = {N}") for N in sizes]
axes[1].legend(handles=handles, loc="lower right", title="markers: VQE\nlines: exact", fontsize=9, title_fontsize=9)
fig.suptitle(r"$\sqrt{\langle M^2 \rangle}$ vs. $J$ for different chain lengths  ($h_x = 1$)")
plt.show()

What the curves show:

- **At $J = 0$** every curve starts at $1/\sqrt N$ (0.500, 0.447, 0.408 and 0.354 for $N$ = 4, 5, 6 and 8): the spins are completely random.
- **The curves get steeper as $N$ grows.** Below the transition, longer chains give smaller values (at $J = 0.5$: 0.600, 0.545, 0.503, 0.441); above it, longer chains give larger values (at $J = 2$: 0.935, 0.942, 0.946, 0.950). This is how a phase transition shows up in finite systems: the change becomes sharper and sharper as the system grows, and for an infinitely long chain it becomes a sudden jump at $J/h_x = 1$.
- **VQE agrees well with the exact results** away from the transition. Near $J \approx 1$ the differences reach about 0.13 for $N = 8$: compare with your answer to Question 4. Exercise 2 below lets you test your explanation.

## 8. Going further: the Binder cumulant, exact solution vs VQE (optional)

The curves above show *that* the transition sharpens, but not exactly *where* it is. A standard trick, invented for computer simulations of magnets, is the **Binder cumulant**

$$U_4 = 1 - \frac{\langle M^4\rangle}{3\langle M^2\rangle^2}.$$

It is close to 0 when the spins are disordered and close to $2/3$ when they are ordered. At the transition it hardly depends on the length of the chain, so the curves for different $N$ **cross** there.

We compute $U_4$ in two ways and compare them:

1. **Exactly**, from the exact ground states (Section 5), for chains of up to 10 qubits and many values of $J$. This is a classical calculation, possible only for short chains.
2. **From the VQE states** of Sections 6 and 7, as a quantum computer would do it: the Estimator measures $\langle M^2\rangle$ and $\langle M^4\rangle$ on the circuit with the optimized angles. We don't need to run VQE again: `scan[N]["runs"]` keeps, for each $J$, the angles of the three runs, the best one first. $M^4 = M^2 M^2$ contains only $Z$s, like $M^2$, so on a real device both come from the same measurements.

> **Hint: two kinds of state, two tools.**
>
> - The **exact ground state** is just a vector of $2^N$ amplitudes, returned by `exact_ground_state`: no circuit prepares it. To compute expectation values in it, wrap it in a `Statevector` and use its `expectation_value` method. The result is a complex number with zero imaginary part, so take `.real`: `Statevector(psi).expectation_value(M2).real`.
> - A **VQE state** is *prepared by a circuit* with the optimized angles, as on a quantum computer, so you measure it with the Estimator, as in Sections 6 and 7. One call measures both observables, and the values are already real: `estimator.run([(circuit, [M2, M4], angles)]).result()[0].data.evs`.

Each of the next two cells takes a few seconds.

#### Task 8 (optional): the exact Binder cumulant

Compute $U_4$ in the exact ground state, for each chain length in `binder_sizes` and each value of $J$ in `J_fine`. Store the results in the dictionary `binder`: `binder[N]` must be a NumPy array with one value for each $J$ in `J_fine`, in the same order. The plot below uses these names.

In [ ]:
# 1. Exact U_4, from the exact ground states, on a fine grid of J and for chains of up to 10 qubits
J_fine = np.linspace(0.25, 2.5, 91)
binder_sizes = [4, 6, 8, 10]

binder = {}  # binder[N]: NumPy array with U_4 for each J in J_fine
# YOUR CODE HERE

In [ ]:
check_binder_exact(J_fine, binder_sizes, binder, h_x)

#### Task 9 (optional): the Binder cumulant of the VQE states

Compute $U_4$ in the VQE states of Sections 6 and 7, for each chain length in `vqe_sizes` and each value of $J$ in `J_list`, as a quantum computer would. Don't run VQE again: the optimized angles are stored in `scan`. Store the results in the dictionary `vqe_binder`, with the same structure as `binder`: `vqe_binder[N][k]` is $U_4$ at `J_list[k]`. The table and the plot below use these names.

In [ ]:
# 2. U_4 from the VQE states of Sections 6 and 7, without running VQE again
vqe_sizes = [4, 6, 8]

vqe_binder = {}  # vqe_binder[N]: NumPy array with U_4 for each J in J_list
# YOUR CODE HERE

In [ ]:
check_binder_vqe(J_list, vqe_sizes, vqe_binder, h_x)

print("\nBinder cumulant U_4 from the VQE states (compare with the exact curves in the plot below)\n")
print(f"{'J':>5}" + "".join(f"{'N = ' + str(N):>9}" for N in vqe_sizes))
for k, J in enumerate(J_list):
    print(f"{J:5.2f}" + "".join(f"{vqe_binder[N][k]:9.3f}" for N in vqe_sizes))

In [ ]:
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(7.5, 4.8))
J_arr = np.array(J_list)
shown = (J_arr >= J_fine[0]) & (J_arr <= J_fine[-1])  # the values of J inside the range of the exact curves
for N in binder_sizes:
    ax.plot(J_fine, binder[N], color=SIZE_COLORS[N], lw=1.5, alpha=0.7)  # exact
for N in vqe_sizes:
    ax.plot(J_arr[shown], vqe_binder[N][shown], ls="none", marker=SIZE_MARKERS[N], ms=8, color=SIZE_COLORS[N])  # VQE
ax.axvline(1.0, color="0.5", lw=1, ls=":")
ax.axhline(2 / 3, color="0.6", lw=1, ls="--")
ax.text(0.27, 2 / 3 + 0.006, "ordered limit 2/3", ha="left", va="bottom", color="0.35", fontsize=9)
ax.set_ylim(0.08, 0.71)
ax.set_xlabel(r"$J/h_x$")
ax.set_ylabel(r"$U_4$")
ax.set_title("Binder cumulant: exact ground states and VQE states")
handles = [Line2D([], [], color=SIZE_COLORS[N], marker=SIZE_MARKERS[N] if N in vqe_sizes else None, lw=1.5,
                  label=f"N = {N}") for N in binder_sizes]
ax.legend(handles=handles, loc="lower right", title="lines: exact\nmarkers: VQE", fontsize=9, title_fontsize=9)
plt.show()

What the two calculations show:

- **The exact curves cross close to $J/h_x = 1$**: a little above 1 for the shortest chains, and closer to 1 for the longest ones (look where the $N = 8$ and $N = 10$ curves meet). With longer and longer chains the crossings would converge to the exact value, $J/h_x = 1$.
- **Far from the transition, VQE agrees with the exact results.** At $J = 0$ the VQE values are 0.167, 0.111 and 0.083: exactly $2/(3N)$, the value for completely random spins. At $J = 2$ the VQE points sit almost on the exact curves.
- **Just below the transition the VQE values are too low, and more so for longer chains.** At $J = 1$ VQE gives 0.432 for $N = 4$, a little below the exact curve (about 0.47), but 0.295 for $N = 8$, far below it (about 0.44). With a single layer of CNOTs the circuit cannot build all the correlations between distant spins (Section 7), and $U_4$ is affected even more than $\sqrt{\langle M^2\rangle}$.
- **At $J = 1.25$ the VQE values for $N = 6$ and 8 jump above the exact curves** (0.571 and 0.595, where the exact curves are at about 0.54 and 0.55). There the optimizer has found a state that is more ordered than the true ground state: the same jump appears in $\sqrt{\langle M^2\rangle}$ in the table of Section 7.
- **So the VQE curves cross too, between $J = 1.1$ and $1.25$, but because of this jump**, not through the smooth curves that the method relies on. With VQE points only every 0.1 to 0.25 in $J$, we also cannot place the crossing as precisely as with the exact curves.

**A deeper circuit fixes this.** With three layers of rotations and CNOTs (Qiskit's `efficient_su2(N, su2_gates=["rz", "rx", "rz"], entanglement="linear", reps=3)`, with $12N$ angles), the VQE values fall on the exact curves: at $J = 1$, for example, $N = 8$ gives 0.425, close to the exact 0.44, and the curves cross smoothly between $J = 1.1$ and 1.25. The price is time: the scans of Section 7 take about half an hour instead of two to three minutes, which is why we use a single layer here. Exercise 2 lets you try a deeper circuit.

## 9. Supplementary: from a chain to a square lattice (for those who finish)

If you have finished, adapt what you have done to the **two-dimensional** Ising model: $N = L \times L$ qubits on a square lattice, each interacting with its neighbours on the left, on the right, above and below. The Hamiltonian has the same form as before,

$$H = -J \sum_{\langle i,j\rangle} Z_i Z_j \;-\; h_x \sum_{i=0}^{N-1} X_i ,$$

but the first sum now runs over all the pairs $\langle i,j\rangle$ of neighbouring sites, the *bonds* of the lattice, horizontal and vertical. The transition moves too: for an infinitely large lattice it is at $J/h_x \approx 0.33$ (the critical field is $h_x \approx 3.04\,J$) [6], instead of 1. In small lattices, as in short chains, the change is gradual and appears at somewhat larger values of $J$.

You already have almost all the pieces. Some hints:

1. **Number the qubits.** Give the site in row $r$ and column $c$ the index $i = rL + c$. For $L = 3$:
   ```
   0 — 1 — 2
   |   |   |
   3 — 4 — 5
   |   |   |
   6 — 7 — 8
   ```
2. **List the bonds.** Write a function `lattice_bonds(L)` that returns the pairs `(i, j)` of neighbouring qubits: for each site, its neighbour on the right (unless the site is in the last column) and the one below (unless it is in the last row). With open boundaries, as for our chain, $L = 2$ has 4 bonds and $L = 3$ has 12.
3. **The Hamiltonian.** Adapt `ising_hamiltonian` into `ising_hamiltonian_2d(L, J, h_x)`, with one $ZZ$ term for each bond. Check it with `exact_ground_state`: for $h_x = 0$ the lowest energy is $-J$ times the number of bonds.
4. **Reuse.** `run_vqe`, the Estimator, `magnetization_op` and `magnetization_squared_op` (with $N = L^2$), and `exact_ground_state` work as they are.
5. **The circuit.** `create_ansatz(L * L)` still works: its CNOTs connect the qubits in the order 0, 1, 2, …, which follows the rows but not the vertical bonds. You can also let the CNOTs follow the lattice: `efficient_su2(L * L, su2_gates=["rz", "rx", "rz"], entanglement=lattice_bonds(L), reps=1)` puts one CNOT on each bond. Try both.
6. **The scan.** Start with $L = 2$ (4 qubits), then $L = 3$ (9 qubits). Put more values of $J$ where the transition is. Larger lattices are too slow for VQE here: $L = 4$ already has 16 qubits. Compare with the exact results, as in Section 6.

In [ ]:
# Your 2D version: start with L = 2, then try L = 3
L = 2
N = L * L

# YOUR CODE HERE

## 10. What changes on a real quantum computer?

Everything quantum in this notebook goes through the Estimator (apart from the look at the three-qubit states in Section 3, which only a simulator allows), so the same code can run on real hardware by replacing the Aer Estimator with the Estimator of a quantum computer (for IBM devices, from the `qiskit-ibm-runtime` package). A few things change:

- the circuit must first be **transpiled**: rewritten in the device's native gates and mapped onto its physical qubits;
- each expectation value is estimated from a finite number of measurements (**shots**), so it comes with statistical noise, on top of the noise of the device itself;
- optimizers that cope better with noisy measurements, such as SPSA or COBYLA, are then preferred to L-BFGS-B.

Measuring $M^2$ costs nothing extra: it contains only $Z$s, so it comes from the same measurements as the $ZZ$ terms of the energy.

## Summary

- The **Ising model in a transverse field** has two phases: ordered (spins aligned along $z$) when $J \gg h_x$, and disordered when $h_x \gg J$. For a long chain, the quantum phase transition between them is at $J/h_x = 1$.
- **VQE** finds the ground state by minimizing the energy of a parameterized circuit. Here: the hardware-efficient ansatz, energies and gradients from Qiskit's Estimator, and SciPy's L-BFGS-B optimizer started from three random points.
- The **size of the magnetization**, $\sqrt{\langle M^2\rangle}$, tracks the order. Its curves get steeper as the chain grows: the finite-size signature of the transition. The Binder cumulant locates it close to $J/h_x = 1$.

## More exercises (if you have time)


1. **A better circuit.** Give the ansatz two layers: repeat steps 2 and 3 of `create_ansatz` with new angles, or use Qiskit's `n_local(N, ["rz", "rx", "rz"], "cx", entanglement="linear", reps=2)` from `qiskit.circuit.library`. Does VQE get closer to the exact curve near $J = 1$? (The runs take longer.)
2. **A different field.** Set $h_x = 0.5$ and repeat the scan for $N = 5$. Where do you expect the transition now? Does the curve agree?
3. **A ring instead of a line.** Add the bond between the last and the first qubit to the Hamiltonian (and a CNOT between them in the ansatz). How do the results change?

## References

1. D. Pope and T. Shah, ["Seeing quantum phase transitions with quantum computers"](https://pennylane.ai/qml/demos/tutorial_quantum_phase_transitions), PennyLane demo (2025).
2. A. Kandala et al., "Hardware-efficient Variational Quantum Eigensolver for Small Molecules and Quantum Magnets", [arXiv:1704.05018](https://arxiv.org/abs/1704.05018) (2017).
3. S. Sachdev, *Quantum Phase Transitions*, 2nd ed., Cambridge University Press (2011).
4. P. Pfeuty, "The one-dimensional Ising model with a transverse field", Ann. Phys. **57**, 79 (1970).
5. K. Binder, "Finite size scaling analysis of Ising model block distribution functions", Z. Phys. B **43**, 119 (1981).